In [3]:
%load_ext autoreload

In [4]:
%autoreload 2
from util_libs.utils import (
    select_users_by_period,
    create_hourly_user_dataset,
)
from util_libs.visualization_utils import (
    plot_user_metrics,
    plot_market_features,
)
from util_libs.spikes_utils import (
    detect_market_spikes,
)

### Prepare data

In [1]:
import pandas as pd
import numpy as np
import json
pd.set_option('display.max_columns', 500)

MARKET = "eth_cbbtc_usdc"
# MARKET = "eth_mF-ONE_usdc"
# MARKET = "eth_susde_pyusd"
EVENTS_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_enriched"
HOURLY_MARKET_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_hourly_data"
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv(f"{EVENTS_PATH}/{MARKET}.csv")
market_df = pd.read_csv(f"{HOURLY_MARKET_PATH}/{MARKET}.csv")

df.head(2)

,hash,type,timestamp,user_address,assets,assets_usd,liquidated_assets,liquidated_assets_usd,market,datetime,market_address,total_supply_before,total_borrow_before,total_supply_after,total_borrow_after,utilization_before,utilization_after,tx_actions,borrow_rate_before,supply_rate_before,borrow_rate_after,supply_rate_after,collateral_price,loan_asset_price,collateral_before,collateral_value_before,debt_before,supply_before,ltv_before,collateral_after,collateral_value_after,debt_after,supply_after,ltv_after,health_factor_before,health_factor_after,event_type,vault_flg,volatility_1h,drawdown_1h,trend_1h,volatility_6h,drawdown_6h,trend_6h,volatility_24h,drawdown_24h,trend_24h
0,0x55b1f9bfe77fa41b9f74b6949024ed40c61fda35554c...,MarketSupply,1726145495,0x29d4CDFee8F533af8529A9e1517b580E022874f7,1000000,0.999934,0,0.0,eth_cbbtc_usdc,2024-09-12 12:51:35,0x64d65c9a2d91c36d56fbc42d69e979335320169b3df6...,0.0,0.0,1.0,0.89,0.0,0.89,3,0.003503,0.0,0.013895,0.012367,NaN,0.999641,0.0,NaN,0.0,0.0,0.0,0.00005,NaN,0.88968,0.999641,0.0,0.0,0.0,position_open,False,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
1,0x55b1f9bfe77fa41b9f74b6949024ed40c61fda35554c...,MarketSupplyCollateral,1726145495,0x29d4CDFee8F533af8529A9e1517b580E022874f7,5000,2.903950,0,0.0,eth_cbbtc_usdc,2024-09-12 12:51:35,0x64d65c9a2d91c36d56fbc42d69e979335320169b3df6...,0.0,0.0,1.0,0.89,0.0,0.89,3,0.003503,0.0,0.013895,0.012367,NaN,0.999641,0.0,NaN,0.0,0.0,0.0,0.00005,NaN,0.88968,0.999641,0.0,0.0,0.0,position_open,False,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:

plot_market_features(
    market_df,
    ["utilization", "total_borrow"],
    y_ranges={
        "total_supply": [0,200_000_000],
        "total_borrow": [0,200_000_000],
        "utilization": [0.6, 1]
    }
)

In [34]:
spikes = detect_market_spikes(
    df,
    start_date="2026-01-15",
    spike_util_threshold=0.91,
)
len(spikes)

21

In [35]:
for i in spikes[:1550]:
    df_actions = i['actions_df'].copy()
    if df_actions.empty:
        continue
    
    trigger_ts = i['trigger_datetime']
    # df_actions = df_actions[df_actions['datetime'] > trigger_ts]
    if df_actions.empty:
        continue
    display(df_actions[[
        "datetime",
        "type",
        "utilization_before",
        "utilization_after",
        "borrow_rate_after",
        "user_address"
    ]])
    print("=" * 250)


,datetime,type,utilization_before,utilization_after,borrow_rate_after,user_address
77,2026-01-22 15:10:11,MarketWithdraw,0.878083,1.000000,0.124413,0x19b3cD7032B8C062E8d44EaCad661a0970DD8c55
78,2026-01-22 15:41:11,MarketSupply,1.000000,0.900001,0.031104,0x19b3cD7032B8C062E8d44EaCad661a0970DD8c55


,datetime,type,utilization_before,utilization_after,borrow_rate_after,user_address
79,2026-01-22 19:27:23,MarketWithdraw,0.900001,0.956299,0.083879,0x19b3cD7032B8C062E8d44EaCad661a0970DD8c55


,datetime,type,utilization_before,utilization_after,borrow_rate_after,user_address
98,2026-01-28 01:57:35,MarketWithdraw,0.900001,0.999999,0.13029,0x19b3cD7032B8C062E8d44EaCad661a0970DD8c55
99,2026-01-28 05:29:59,MarketWithdrawCollateral,0.999999,0.901043,0.03418,0xb0f8C20b849886cAD270D705Acc39739c0683f64


,datetime,type,utilization_before,utilization_after,borrow_rate_after,user_address
101,2026-01-31 10:01:59,MarketBorrow,0.901043,0.937408,0.070859,0xb00FE7A3e0EAA061dE1f74Af0D48dE97c244F58d


,datetime,type,utilization_before,utilization_after,borrow_rate_after,user_address
102,2026-01-31 10:01:59,MarketSupplyCollateral,0.901043,0.937408,0.070859,0xb00FE7A3e0EAA061dE1f74Af0D48dE97c244F58d


...

,datetime,type,utilization_before,utilization_after,borrow_rate_after,user_address
345,2026-03-03 06:16:35,MarketBorrow,0.865750,0.929800,0.053889,0x9F1a1479191Af103Ff82FeAD23e950B6B3B2bD1a
346,2026-03-03 07:20:11,MarketSupplyCollateral,0.929800,0.956973,0.077170,0xb00FE7A3e0EAA061dE1f74Af0D48dE97c244F58d
347,2026-03-03 07:20:11,MarketBorrow,0.929800,0.956973,0.077170,0xb00FE7A3e0EAA061dE1f74Af0D48dE97c244F58d
348,2026-03-03 07:27:23,MarketSupplyCollateral,0.956973,0.980565,0.097331,0xb00FE7A3e0EAA061dE1f74Af0D48dE97c244F58d
349,2026-03-03 07:27:23,MarketBorrow,0.956973,0.980565,0.097331,0xb00FE7A3e0EAA061dE1f74Af0D48dE97c244F58d
350,2026-03-03 07:37:11,MarketSupplyCollateral,0.980565,1.000000,0.113939,0xb00FE7A3e0EAA061dE1f74Af0D48dE97c244F58d
351,2026-03-03 07:37:11,MarketBorrow,0.980565,1.000000,0.113939,0xb00FE7A3e0EAA061dE1f74Af0D48dE97c244F58d
352,2026-03-03 07:41:11,MarketSupplyCollateral,1.000000,1.000000,0.113939,0xb00FE7A3e0EAA061dE1f74Af0D48dE97c244F58d
353,2026-03-03 10:47:23,MarketSupply,1.000000,0.998855,0.114702,0x37057Ed5FC01a7Ad3403358026C7419c3E15C004
354,2026-03-03 10:52:47,MarketSupply,0.998855,0.975369,0.094323,0x19b3cD7032B8C062E8d44EaCad661a0970DD8c55


,datetime,type,utilization_before,utilization_after,borrow_rate_after,user_address
395,2026-03-04 09:08:23,MarketBorrow,0.899809,0.979999,0.098997,0x9F1a1479191Af103Ff82FeAD23e950B6B3B2bD1a


,datetime,type,utilization_before,utilization_after,borrow_rate_after,user_address
396,2026-03-04 09:08:23,MarketSupplyCollateral,0.899809,0.979999,0.098997,0x9F1a1479191Af103Ff82FeAD23e950B6B3B2bD1a
397,2026-03-04 09:23:47,MarketSupplyCollateral,0.979999,1.000000,0.116469,0x9F1a1479191Af103Ff82FeAD23e950B6B3B2bD1a
398,2026-03-04 09:23:47,MarketBorrow,0.979999,1.000000,0.116469,0x9F1a1479191Af103Ff82FeAD23e950B6B3B2bD1a
399,2026-03-04 14:00:23,MarketSupply,1.000000,0.999082,0.118885,0x9dEfF269B22849889cAE9A965769f576a1e72d27
400,2026-03-04 14:05:23,MarketWithdraw,0.999082,0.999999,0.119709,0x9dEfF269B22849889cAE9A965769f576a1e72d27
401,2026-03-04 14:10:35,MarketSupply,0.999999,0.908995,0.038004,0x2C793f5cB25B35A99648783c01E6cCCC200D2096


,datetime,type,utilization_before,utilization_after,borrow_rate_after,user_address
712,2026-04-08 07:52:59,MarketWithdraw,0.909083,0.999652,0.094800,0x19b3cD7032B8C062E8d44EaCad661a0970DD8c55
713,2026-04-08 09:23:11,MarketRepay,0.999652,0.971532,0.075231,0x502D222e8e4DaEF69032f55F0c1A999EFFd78fB3
714,2026-04-08 09:23:11,MarketWithdrawCollateral,0.999652,0.971532,0.075231,0x502D222e8e4DaEF69032f55F0c1A999EFFd78fB3
715,2026-04-08 09:31:23,MarketRepay,0.971532,0.943437,0.055076,0x502D222e8e4DaEF69032f55F0c1A999EFFd78fB3
716,2026-04-08 09:31:23,MarketWithdrawCollateral,0.971532,0.943437,0.055076,0x502D222e8e4DaEF69032f55F0c1A999EFFd78fB3
717,2026-04-08 09:38:35,MarketRepay,0.943437,0.930011,0.045443,0x502D222e8e4DaEF69032f55F0c1A999EFFd78fB3
718,2026-04-08 09:38:35,MarketWithdrawCollateral,0.943437,0.930011,0.045443,0x502D222e8e4DaEF69032f55F0c1A999EFFd78fB3
719,2026-04-08 13:11:47,MarketSupply,0.930011,0.903465,0.026633,0x19b3cD7032B8C062E8d44EaCad661a0970DD8c55


,datetime,type,utilization_before,utilization_after,borrow_rate_after,user_address
934,2026-04-18 20:14:47,MarketWithdraw,0.923308,1.000000,0.095151,0x19b3cD7032B8C062E8d44EaCad661a0970DD8c55
935,2026-04-18 21:18:11,MarketRepay,1.000000,0.975101,0.077719,0x502D222e8e4DaEF69032f55F0c1A999EFFd78fB3
936,2026-04-18 21:18:11,MarketWithdrawCollateral,1.000000,0.975101,0.077719,0x502D222e8e4DaEF69032f55F0c1A999EFFd78fB3
937,2026-04-18 21:18:59,MarketWithdraw,0.975101,0.999460,0.095178,0x19b3cD7032B8C062E8d44EaCad661a0970DD8c55
938,2026-04-18 21:19:59,MarketWithdraw,0.999460,1.000000,0.095565,0x19b3cD7032B8C062E8d44EaCad661a0970DD8c55
939,2026-04-19 00:46:11,MarketWithdrawCollateral,1.000000,0.999851,0.097105,0x7e68c279EA86FA49A49Eef2Cbb79B9cBfBc48025
940,2026-04-19 00:46:11,MarketRepay,1.000000,0.999851,0.097105,0x7e68c279EA86FA49A49Eef2Cbb79B9cBfBc48025
941,2026-04-19 00:46:23,MarketWithdraw,0.999851,1.000000,0.097213,0x19b3cD7032B8C062E8d44EaCad661a0970DD8c55
942,2026-04-19 01:58:47,MarketRepay,1.000000,0.999790,0.097616,0x6A553aD3C49E2Fb7d745F04599eCd47CFC1b3C2a
943,2026-04-19 01:58:47,MarketWithdrawCollateral,1.000000,0.999790,0.097616,0x6A553aD3C49E2Fb7d745F04599eCd47CFC1b3C2a


In [10]:
df.groupby("user_address")["debt_after"].max().sort_values(ascending=False)[:30]

user_address
0x64471d103A7f77262529383D53Bdd28b260B1aE8    4.219167e+07
0x50B82f42D51D875b1bfdd946ad27AeE2a6b1AB41    1.172820e+07
0x14bCD9da052Cdc6fE0b9446d5a616D5b7B4d4550    1.059576e+07
0x1597E4B7cF6D2877A1d690b6088668afDb045763    1.038704e+07
0xBccbc9fC2d1D9e9aBd0Ca132BE816bd4f6051B86    5.440278e+06
0xc9a12aBCa328b6903a755080befFaB602B20c028    4.534564e+06
0xa33E1f748754D2d624638ab335100d92fcBe62A2    3.044245e+06
0x20b20eAE302c821b53018037B0f3c1eC90c0af5B    2.560799e+06
0xBf6D2A1e5C16bBcd455a9100684a10cee204fC0c    2.559693e+06
0xeB0Cda1a52f9ac0bD5293bd65b52eD07168e1E8e    1.999940e+06
0xeD8CF2891d7bd5B01a8eE5B702b73b39B1967968    1.339785e+06
0xBBfe856D7ab158a2137cf2CBDf6CeeeE75294Eff    1.018285e+06
0xd682Bae394e04a0A30040D065E5A75529E5e987d    8.807681e+05
0x78f4564bF45254c1D29B65180b1949F372A812f8    7.329178e+05
0x1cF4E7770a6d986673116EAe4f3E3DDcd0964275    6.498888e+05
0x3A6132195d01EB483a20C138bC356E6bD8254EdD    5.549251e+05
0xac97bcA292841079700d67bdAbA3af2a75646097 

In [16]:
hourly = create_hourly_user_dataset(
    df[df["user_address"] == "0x1597E4B7cF6D2877A1d690b6088668afDb045763"],
    market_df,
    1,
)
hourly['liquidity'] = hourly['total_supply'] - hourly['total_borrow']
plot_user_metrics(
    hourly,
    ["debt", "liquidity"],
    "0x1597E4B7cF6D2877A1d690b6088668afDb045763",
)

User address {'0x1597E4B7cF6D2877A1d690b6088668afDb045763'}
MAX DEBT {10387036.90682405}
0.0
10387036.90682405


### Why does total borrow + supply increase simultaneously?